<h1>🧬 Biofilter — Report: <code>annotation_master_gene</code></h1>

Everything the bundle knows about a list of genes, one row per input:
canonical IDs, HGNC metadata, build 38 coordinates, relationship counts
by related entity group, and the number of variants inside the gene's
range.

Reads the bundle natively (ADR-004). Accepts symbols, aliases, synonyms
or cross-reference codes, matched case-insensitively.

### 1. Open a bundle

In [ ]:
from biofilter import Biofilter

BUNDLE = "/path/to/biofilter_data/bundles/20260914"
REPORT = "annotation_master_gene"

bf = Biofilter(db_uri=f"parquet://{BUNDLE}", debug_mode=False)
bf

### 2. What the report offers

In [ ]:
print("columns:")
for column in bf.report.available_columns(REPORT):
    print(" ", column)

print("\nexample input:")
print(bf.report.example_input(REPORT))

In [ ]:
print(bf.report.explain(REPORT))

### 3. Run it

Any of these resolve to the same gene — symbol, synonym, or code:

```
TP53   p53   HGNC:11998   ENSG00000141510   7157
```

In [ ]:
input_genes = [
    "TP53",
    "BRCA1",
    "ENSG00000146648",   # EGFR, by Ensembl id
    "NOT_A_GENE",        # kept in the output, with status='not_found'
]

result = bf.report.run(
    REPORT,
    input_data=input_genes,
    include_relationships=True,
    include_variant_summary=True,
    emit_not_found_rows=True,
)

df = result.to_pandas()
print(f"{result.num_rows} rows from bundle {result.provenance['bundle_id']}")
df[["input_value", "input_matched_alias", "gene_symbol", "entity_id", "status"]]

### 4. Reading the result

`status` is the first column to look at.

| value | meaning |
| --- | --- |
| `ok` | resolved, with a gene record and build 38 coordinates |
| `partial` | resolved, but something is missing — `note` says what |
| `not_found` | the bundle has no gene entity for this input |

Unresolved inputs are **kept on purpose**. Dropping them would leave no
way to tell "absent from this bundle" from "never asked for".

In [ ]:
df[["input_value", "status", "note"]]

#### Identity and coordinates

In [ ]:
df[[
    "input_value",
    "gene_symbol",
    "hgnc_id",
    "ensembl_id",
    "entrez_id",
    "hgnc_status",
    "omic_status",
    "gene_locus_group",
    "build",
    "chromosome",
    "start_position",
    "end_position",
]]

#### Lists: groups, relationships, other aliases

Three columns hold lists rather than scalars. In a DataFrame and in
parquet they are real lists; exported to CSV they are written as JSON so
one cell can hold them.

In [ ]:
for _, row in df[df["status"] != "not_found"].iterrows():
    print(row["gene_symbol"])
    print("  gene groups :", list(row["gene_groups"]))
    print("  relationships:", row["total_entity_relationships"], "total")
    for entry in row["entity_relationships_by_group"]:
        print(f"      {entry['group_name']:<12} {entry['count']:>6}")
    print("  other aliases:", list(row["other_aliases"])[:6])
    print()

Relationships are counted **in both directions**: a gene appearing on
either side of a relationship counts it, grouped by what is on the other
side.

#### `variant_count_in_gene_range`: null is not zero

| value | meaning |
| --- | --- |
| a number | the range was searched, and held that many variants |
| `0` | the range was searched and held none |
| `NaN` / null | the count was **not made** |

Null happens when the gene has no build 38 range, or when
`include_variant_summary=False`.

⚠️ A bundle built for a subset of chromosomes returns `0` for every gene
outside them. That is true of the bundle, not of the genome — check what
the bundle covers before reading a zero as biology.

In [ ]:
df[["input_value", "chromosome", "start_position", "end_position",
    "variant_count_in_gene_range"]]

### 5. Lighter modes

`include_relationships` and `include_variant_summary` are the two
expensive sections. Turning them off is what makes whole-catalog mode
comfortable.

In [ ]:
fast = bf.report.run(
    REPORT,
    input_data=input_genes,
    include_relationships=False,
    include_variant_summary=False,
)

fast.to_pandas()[["input_value", "gene_symbol", "hgnc_id", "chromosome", "status"]]

### 6. Every gene in the bundle

`input_data="__ALL__"` annotates every gene entity instead of a list.

In [ ]:
import time

started = time.perf_counter()
everything = bf.report.run(REPORT, input_data="__ALL__")
elapsed = time.perf_counter() - started

catalog = everything.to_pandas()
print(f"{everything.num_rows:,} genes in {elapsed:.1f}s")
print(catalog["status"].value_counts().to_dict())

In [ ]:
# Genes with no build 38 location are the 'partial' ones, and they are
# also exactly the rows whose variant count is null.
partial = catalog[catalog["status"] == "partial"]
print(f"{len(partial):,} without a build 38 location")
print(f"{catalog['variant_count_in_gene_range'].isna().sum():,} with a null variant count")

In [ ]:
# What the bundle actually covers, which is what a zero above means.
covered = catalog.dropna(subset=["variant_count_in_gene_range"])
covered.groupby("chromosome")["variant_count_in_gene_range"].agg(
    genes="size", with_variants=lambda s: int((s > 0).sum())
).sort_values("with_variants", ascending=False).head(10)

### 7. Export

CSV is the default. The `.provenance.json` written beside it records
which bundle the ids came from — necessary, because `entity_id` means a
different gene in the next build.

In [ ]:
for path in everything.write("annotation_master_gene.csv"):
    print(path)

In [ ]:
# Parquet keeps the list columns as lists, and carries the provenance in
# the file's own metadata.
everything.write("annotation_master_gene.parquet")

### 8. The same thing on the command line

```bash
biofilter --bundle /path/to/bundles/20260914 report run \
    --report-name annotation_master_gene \
    --input TP53 --input BRCA1 \
    --param include_variant_summary=false \
    --output genes.csv
```

`--input-file genes.txt` takes one value per line.

### 9. Quick QA

In [ ]:
expected = list(bf.report.available_columns(REPORT))
missing = [c for c in expected if c not in df.columns]

print("missing columns:", missing or "none")
print("unresolved inputs:", int((df["status"] == "not_found").sum()))
print("bundle:", result.provenance["bundle_id"])
display(df.dtypes.to_frame("dtype"))